Pyspark Assignment 1
# Abhishek_jadhav_69137
# Customer Data Cleaning and Data Wrangling using PySpark

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    trim,
    lower,
    upper,
    regexp_replace,
    when,
    sum
)

spark = SparkSession.builder \
    .appName("CustomerDataCleansingProject") \
    .master("local[*]") \
    .getOrCreate()

print("Spark Session Created")
print("Spark Version:", spark.version)

Spark Session Created
Spark Version: 3.5.9


In [42]:
data = [
    (1001, "John Smith", "Regular", " JOHN.SMITH@ABC.COM ", "9876543210", "Vadodara", " Gujarat ", 65000),
    (1002, "MARY JOHNSON", "premium", "mary.johnson@abc.com", "9876543211", "Mumbai", "Maharashtra", 85000),
    (1003, "David Brown", "Regular", "DAVID.BROWN@ABC.COM", "9876543212", "Ahmedabad", "Gujarat", 72000),
    (1003, "David Brown", "Regular", "DAVID.BROWN@ABC.COM", "9876543212", "Ahmedabad", "Gujarat", 72000),
    (1004, "Lisa Wilson", None, "lisa.wilson@abc.com", "9876543213", "Surat", "Gujarat", None),
    (1005, None, "regular", "james@abc", "9876543214", "Vadodara", "Gujarat", 62000),
    (1006, "Robert Taylor", "Regular", "robert.taylor@abc.com", "9876543215", "Pune", "Maharashtra", 65000),
    (1007, "Susan Lee", "premium", None, "9876543216", "Rajkot", "Gujarat", 58000),
    (1008, "Peter Adams", "Premium", "peter.adams@abc.com", None, "Delhi", "Delhi", 72000)
]

columns = [
    "customer_id",
    "customer_name",
    "customer_type",
    "email",
    "phone",
    "city",
    "state",
    "salary"
]

df = spark.createDataFrame(data, columns)

print(" RAW CUSTOMER DATA ")

df.show(truncate=False)

 RAW CUSTOMER DATA 
+-----------+-------------+-------------+---------------------+----------+---------+-----------+------+
|customer_id|customer_name|customer_type|email                |phone     |city     |state      |salary|
+-----------+-------------+-------------+---------------------+----------+---------+-----------+------+
|1001       |John Smith   |Regular      | JOHN.SMITH@ABC.COM  |9876543210|Vadodara | Gujarat   |65000 |
|1002       |MARY JOHNSON |premium      |mary.johnson@abc.com |9876543211|Mumbai   |Maharashtra|85000 |
|1003       |David Brown  |Regular      |DAVID.BROWN@ABC.COM  |9876543212|Ahmedabad|Gujarat    |72000 |
|1003       |David Brown  |Regular      |DAVID.BROWN@ABC.COM  |9876543212|Ahmedabad|Gujarat    |72000 |
|1004       |Lisa Wilson  |NULL         |lisa.wilson@abc.com  |9876543213|Surat    |Gujarat    |NULL  |
|1005       |NULL         |regular      |james@abc            |9876543214|Vadodara |Gujarat    |62000 |
|1006       |Robert Taylor|Regular      |rob

In [44]:
print(" CUSTOMER DATA SCHEMA ")

df.printSchema()

 CUSTOMER DATA SCHEMA 
root
 |-- customer_id: long (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- customer_type: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- salary: long (nullable = true)



In [45]:
print("Total Records:", df.count())

Total Records: 9


In [46]:
print(" SAMPLE CUSTOMER RECORDS ")
df.show(truncate=False)

 SAMPLE CUSTOMER RECORDS 
+-----------+-------------+-------------+---------------------+----------+---------+-----------+------+
|customer_id|customer_name|customer_type|email                |phone     |city     |state      |salary|
+-----------+-------------+-------------+---------------------+----------+---------+-----------+------+
|1001       |John Smith   |Regular      | JOHN.SMITH@ABC.COM  |9876543210|Vadodara | Gujarat   |65000 |
|1002       |MARY JOHNSON |premium      |mary.johnson@abc.com |9876543211|Mumbai   |Maharashtra|85000 |
|1003       |David Brown  |Regular      |DAVID.BROWN@ABC.COM  |9876543212|Ahmedabad|Gujarat    |72000 |
|1003       |David Brown  |Regular      |DAVID.BROWN@ABC.COM  |9876543212|Ahmedabad|Gujarat    |72000 |
|1004       |Lisa Wilson  |NULL         |lisa.wilson@abc.com  |9876543213|Surat    |Gujarat    |NULL  |
|1005       |NULL         |regular      |james@abc            |9876543214|Vadodara |Gujarat    |62000 |
|1006       |Robert Taylor|Regular    

In [47]:
print(" NULL COUNTS ")

null_counts = df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])
null_counts.show()

 NULL COUNTS 
+-----------+-------------+-------------+-----+-----+----+-----+------+
|customer_id|customer_name|customer_type|email|phone|city|state|salary|
+-----------+-------------+-------------+-----+-----+----+-----+------+
|          0|            1|            1|    1|    1|   0|    0|     1|
+-----------+-------------+-------------+-----+-----+----+-----+------+



In [48]:
print(" RECORDS WITH NULL VALUES ")

df.filter(
    col("customer_id").isNull() |
    col("customer_name").isNull() |
    col("customer_type").isNull() |
    col("email").isNull() |
    col("phone").isNull() |
    col("salary").isNull()
).show(truncate=False)

 RECORDS WITH NULL VALUES 
+-----------+-------------+-------------+-------------------+----------+--------+-------+------+
|customer_id|customer_name|customer_type|email              |phone     |city    |state  |salary|
+-----------+-------------+-------------+-------------------+----------+--------+-------+------+
|1004       |Lisa Wilson  |NULL         |lisa.wilson@abc.com|9876543213|Surat   |Gujarat|NULL  |
|1005       |NULL         |regular      |james@abc          |9876543214|Vadodara|Gujarat|62000 |
|1007       |Susan Lee    |premium      |NULL               |9876543216|Rajkot  |Gujarat|58000 |
|1008       |Peter Adams  |Premium      |peter.adams@abc.com|NULL      |Delhi   |Delhi  |72000 |
+-----------+-------------+-------------+-------------------+----------+--------+-------+------+



In [49]:
print(" DUPLICATE CUSTOMER IDs ")

df.groupBy(
    "customer_id"
).count().filter(
    col("count") > 1
).show()

 DUPLICATE CUSTOMER IDs 
+-----------+-----+
|customer_id|count|
+-----------+-----+
|       1003|    2|
+-----------+-----+



In [50]:
print("Total Records:", df.count())

print(
    "Unique Customer IDs:",
    df.select("customer_id").distinct().count()
)

print(
    "Duplicate Customer IDs:",
    df.groupBy("customer_id")
      .count()
      .filter(col("count") > 1)
      .count()
)

Total Records: 9
Unique Customer IDs: 8
Duplicate Customer IDs: 1


In [51]:
print(" NULL REPLACEMENT ")

df_filled = df.fillna({
    "customer_type": "UNKNOWN",
    "email": "NOT_AVAILABLE",
    "phone": "NOT_AVAILABLE",
    "salary": 0
})
df_filled.show(truncate=False)

 NULL REPLACEMENT 
+-----------+-------------+-------------+---------------------+-------------+---------+-----------+------+
|customer_id|customer_name|customer_type|email                |phone        |city     |state      |salary|
+-----------+-------------+-------------+---------------------+-------------+---------+-----------+------+
|1001       |John Smith   |Regular      | JOHN.SMITH@ABC.COM  |9876543210   |Vadodara | Gujarat   |65000 |
|1002       |MARY JOHNSON |premium      |mary.johnson@abc.com |9876543211   |Mumbai   |Maharashtra|85000 |
|1003       |David Brown  |Regular      |DAVID.BROWN@ABC.COM  |9876543212   |Ahmedabad|Gujarat    |72000 |
|1003       |David Brown  |Regular      |DAVID.BROWN@ABC.COM  |9876543212   |Ahmedabad|Gujarat    |72000 |
|1004       |Lisa Wilson  |UNKNOWN      |lisa.wilson@abc.com  |9876543213   |Surat    |Gujarat    |0     |
|1005       |NULL         |regular      |james@abc            |9876543214   |Vadodara |Gujarat    |62000 |
|1006       |Rober

In [52]:
print(" REMOVE RECORDS WITH MISSING MANDATORY FIELDS ")

df_valid = df_filled.dropna(
    subset=["customer_id", "customer_name"]
)
df_valid.show(truncate=False)

 REMOVE RECORDS WITH MISSING MANDATORY FIELDS 
+-----------+-------------+-------------+---------------------+-------------+---------+-----------+------+
|customer_id|customer_name|customer_type|email                |phone        |city     |state      |salary|
+-----------+-------------+-------------+---------------------+-------------+---------+-----------+------+
|1001       |John Smith   |Regular      | JOHN.SMITH@ABC.COM  |9876543210   |Vadodara | Gujarat   |65000 |
|1002       |MARY JOHNSON |premium      |mary.johnson@abc.com |9876543211   |Mumbai   |Maharashtra|85000 |
|1003       |David Brown  |Regular      |DAVID.BROWN@ABC.COM  |9876543212   |Ahmedabad|Gujarat    |72000 |
|1003       |David Brown  |Regular      |DAVID.BROWN@ABC.COM  |9876543212   |Ahmedabad|Gujarat    |72000 |
|1004       |Lisa Wilson  |UNKNOWN      |lisa.wilson@abc.com  |9876543213   |Surat    |Gujarat    |0     |
|1006       |Robert Taylor|Regular      |robert.taylor@abc.com|9876543215   |Pune     |Maharashtr

In [53]:
print(" EXACT DUPLICATE REMOVAL USING distinct() ")

df_distinct = df_valid.distinct()
df_distinct.show(truncate=False)

 EXACT DUPLICATE REMOVAL USING distinct() 
+-----------+-------------+-------------+---------------------+-------------+---------+-----------+------+
|customer_id|customer_name|customer_type|email                |phone        |city     |state      |salary|
+-----------+-------------+-------------+---------------------+-------------+---------+-----------+------+
|1001       |John Smith   |Regular      | JOHN.SMITH@ABC.COM  |9876543210   |Vadodara | Gujarat   |65000 |
|1002       |MARY JOHNSON |premium      |mary.johnson@abc.com |9876543211   |Mumbai   |Maharashtra|85000 |
|1003       |David Brown  |Regular      |DAVID.BROWN@ABC.COM  |9876543212   |Ahmedabad|Gujarat    |72000 |
|1004       |Lisa Wilson  |UNKNOWN      |lisa.wilson@abc.com  |9876543213   |Surat    |Gujarat    |0     |
|1006       |Robert Taylor|Regular      |robert.taylor@abc.com|9876543215   |Pune     |Maharashtra|65000 |
|1007       |Susan Lee    |premium      |NOT_AVAILABLE        |9876543216   |Rajkot   |Gujarat    |58

In [54]:
print(" DUPLICATE REMOVAL USING customer_id ")

df_deduplicated = df_valid.dropDuplicates(
    ["customer_id"]
)
df_deduplicated.show(truncate=False)

 DUPLICATE REMOVAL USING customer_id 
+-----------+-------------+-------------+---------------------+-------------+---------+-----------+------+
|customer_id|customer_name|customer_type|email                |phone        |city     |state      |salary|
+-----------+-------------+-------------+---------------------+-------------+---------+-----------+------+
|1001       |John Smith   |Regular      | JOHN.SMITH@ABC.COM  |9876543210   |Vadodara | Gujarat   |65000 |
|1002       |MARY JOHNSON |premium      |mary.johnson@abc.com |9876543211   |Mumbai   |Maharashtra|85000 |
|1003       |David Brown  |Regular      |DAVID.BROWN@ABC.COM  |9876543212   |Ahmedabad|Gujarat    |72000 |
|1004       |Lisa Wilson  |UNKNOWN      |lisa.wilson@abc.com  |9876543213   |Surat    |Gujarat    |0     |
|1006       |Robert Taylor|Regular      |robert.taylor@abc.com|9876543215   |Pune     |Maharashtra|65000 |
|1007       |Susan Lee    |premium      |NOT_AVAILABLE        |9876543216   |Rajkot   |Gujarat    |58000 |

In [55]:
print(" CLEAN CUSTOMER NAMES ")

df_name_clean = df_deduplicated.withColumn(
    "customer_name",
    regexp_replace(
        trim(col("customer_name")),
        r"\s+",
        " "
    )
)

df_name_clean.select(
    "customer_id",
    "customer_name"
).show(truncate=False)

 CLEAN CUSTOMER NAMES 
+-----------+-------------+
|customer_id|customer_name|
+-----------+-------------+
|1001       |John Smith   |
|1002       |MARY JOHNSON |
|1003       |David Brown  |
|1004       |Lisa Wilson  |
|1006       |Robert Taylor|
|1007       |Susan Lee    |
|1008       |Peter Adams  |
+-----------+-------------+



In [56]:
print(" STANDARDIZE CUSTOMER TYPE ")

df_type_clean = df_name_clean.withColumn(
    "customer_type",
    upper(trim(col("customer_type")))
)

df_type_clean.select(
    "customer_id",
    "customer_type"
).show()

 STANDARDIZE CUSTOMER TYPE 
+-----------+-------------+
|customer_id|customer_type|
+-----------+-------------+
|       1001|      REGULAR|
|       1002|      PREMIUM|
|       1003|      REGULAR|
|       1004|      UNKNOWN|
|       1006|      REGULAR|
|       1007|      PREMIUM|
|       1008|      PREMIUM|
+-----------+-------------+



In [57]:
print(" STANDARDIZE EMAIL ")

df_email_clean = df_type_clean.withColumn(
    "email",
    lower(trim(col("email")))
)
df_email_clean.select(
    "customer_id",
    "email"
).show(truncate=False)

 STANDARDIZE EMAIL 
+-----------+---------------------+
|customer_id|email                |
+-----------+---------------------+
|1001       |john.smith@abc.com   |
|1002       |mary.johnson@abc.com |
|1003       |david.brown@abc.com  |
|1004       |lisa.wilson@abc.com  |
|1006       |robert.taylor@abc.com|
|1007       |not_available        |
|1008       |peter.adams@abc.com  |
+-----------+---------------------+



In [58]:
print(" CLEAN CITY AND STATE ")

df_location_clean = df_email_clean \
    .withColumn(
        "city",
        trim(col("city"))
    ) \
    .withColumn(
        "state",
        trim(col("state"))
    )
df_location_clean.select(
    "customer_id",
    "city",
    "state"
).show(truncate=False)

 CLEAN CITY AND STATE 
+-----------+---------+-----------+
|customer_id|city     |state      |
+-----------+---------+-----------+
|1001       |Vadodara |Gujarat    |
|1002       |Mumbai   |Maharashtra|
|1003       |Ahmedabad|Gujarat    |
|1004       |Surat    |Gujarat    |
|1006       |Pune     |Maharashtra|
|1007       |Rajkot   |Gujarat    |
|1008       |Delhi    |Delhi      |
+-----------+---------+-----------+



In [59]:
print(" CLEAN PHONE NUMBERS ")

df_phone_clean = df_location_clean.withColumn(
    "phone",
    when(
        col("phone") == "NOT_AVAILABLE",
        "NOT_AVAILABLE"
    ).otherwise(
        regexp_replace(
            col("phone"),
            r"[^0-9]",
            ""
        )
    )
)

df_phone_clean.select(
    "customer_id",
    "phone"
).show(truncate=False)

 CLEAN PHONE NUMBERS 
+-----------+-------------+
|customer_id|phone        |
+-----------+-------------+
|1001       |9876543210   |
|1002       |9876543211   |
|1003       |9876543212   |
|1004       |9876543213   |
|1006       |9876543215   |
|1007       |9876543216   |
|1008       |NOT_AVAILABLE|
+-----------+-------------+



In [60]:
print(" EMAIL VALIDATION ")

df_email_check = df_phone_clean.withColumn(
    "email_status",
    when(
        col("email").rlike(
            r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"
        ),
        "VALID"
    ).otherwise("INVALID")
)
df_email_check.select(
    "customer_id",
    "customer_name",
    "email",
    "email_status"
).show(truncate=False)

 EMAIL VALIDATION 
+-----------+-------------+---------------------+------------+
|customer_id|customer_name|email                |email_status|
+-----------+-------------+---------------------+------------+
|1001       |John Smith   |john.smith@abc.com   |VALID       |
|1002       |MARY JOHNSON |mary.johnson@abc.com |VALID       |
|1003       |David Brown  |david.brown@abc.com  |VALID       |
|1004       |Lisa Wilson  |lisa.wilson@abc.com  |VALID       |
|1006       |Robert Taylor|robert.taylor@abc.com|VALID       |
|1007       |Susan Lee    |not_available        |INVALID     |
|1008       |Peter Adams  |peter.adams@abc.com  |VALID       |
+-----------+-------------+---------------------+------------+



In [61]:
clean_df = df \
    .fillna({
        "customer_type": "UNKNOWN",
        "email": "NOT_AVAILABLE",
        "phone": "NOT_AVAILABLE",
        "salary": 0
    }) \
    .dropna(
        subset=["customer_id", "customer_name"]
    ) \
    .dropDuplicates(["customer_id"]) \
    .withColumn(
        "customer_name",
        regexp_replace(
            trim(col("customer_name")),
            r"\s+",
            " "
        )
    ) \
    .withColumn(
        "customer_type",
        upper(trim(col("customer_type")))
    ) \
    .withColumn(
        "email",
        lower(trim(col("email")))
    ) \
    .withColumn(
        "phone",
        when(
            col("phone") == "NOT_AVAILABLE",
            "NOT_AVAILABLE"
        ).otherwise(
            regexp_replace(
                col("phone"),
                r"[^0-9]",
                ""
            )
        )
    ) \
    .withColumn(
        "city",
        trim(col("city"))
    ) \
    .withColumn(
        "state",
        trim(col("state"))
    )

print(" FINAL CLEAN CUSTOMER DATA ")
clean_df.show(truncate=False)

 FINAL CLEAN CUSTOMER DATA 
+-----------+-------------+-------------+---------------------+-------------+---------+-----------+------+
|customer_id|customer_name|customer_type|email                |phone        |city     |state      |salary|
+-----------+-------------+-------------+---------------------+-------------+---------+-----------+------+
|1001       |John Smith   |REGULAR      |john.smith@abc.com   |9876543210   |Vadodara |Gujarat    |65000 |
|1002       |MARY JOHNSON |PREMIUM      |mary.johnson@abc.com |9876543211   |Mumbai   |Maharashtra|85000 |
|1003       |David Brown  |REGULAR      |david.brown@abc.com  |9876543212   |Ahmedabad|Gujarat    |72000 |
|1004       |Lisa Wilson  |UNKNOWN      |lisa.wilson@abc.com  |9876543213   |Surat    |Gujarat    |0     |
|1006       |Robert Taylor|REGULAR      |robert.taylor@abc.com|9876543215   |Pune     |Maharashtra|65000 |
|1007       |Susan Lee    |PREMIUM      |not_available        |9876543216   |Rajkot   |Gujarat    |58000 |
|1008    

In [62]:
print(" DUPLICATE CUSTOMER ID VALIDATION ")

clean_df.groupBy(
    "customer_id"
).count().filter(
    col("count") > 1
).show()

 DUPLICATE CUSTOMER ID VALIDATION 
+-----------+-----+
|customer_id|count|
+-----------+-----+
+-----------+-----+



In [63]:
print(" FINAL NULL COUNT ")

clean_df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in clean_df.columns
]).show()

 FINAL NULL COUNT 
+-----------+-------------+-------------+-----+-----+----+-----+------+
|customer_id|customer_name|customer_type|email|phone|city|state|salary|
+-----------+-------------+-------------+-----+-----+----+-----+------+
|          0|            0|            0|    0|    0|   0|    0|     0|
+-----------+-------------+-------------+-----+-----+----+-----+------+



In [64]:
print(" CUSTOMER TYPE VALIDATION ")

clean_df.select(
    "customer_type"
).distinct().show()

 CUSTOMER TYPE VALIDATION 
+-------------+
|customer_type|
+-------------+
|      REGULAR|
|      UNKNOWN|
|      PREMIUM|
+-------------+



In [65]:
print(" PHONE NUMBER VALIDATION ")

clean_df.select(
    "customer_id",
    "phone"
).show(truncate=False)

 PHONE NUMBER VALIDATION 
+-----------+-------------+
|customer_id|phone        |
+-----------+-------------+
|1001       |9876543210   |
|1002       |9876543211   |
|1003       |9876543212   |
|1004       |9876543213   |
|1006       |9876543215   |
|1007       |9876543216   |
|1008       |NOT_AVAILABLE|
+-----------+-------------+



In [66]:
print(" FINAL EMAIL VALIDATION ")

clean_df.withColumn(
    "email_status",
    when(
        col("email").rlike(
            r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"
        ),
        "VALID"
    ).otherwise("INVALID")
).select(
    "customer_id",
    "email",
    "email_status"
).show(truncate=False)

 FINAL EMAIL VALIDATION 
+-----------+---------------------+------------+
|customer_id|email                |email_status|
+-----------+---------------------+------------+
|1001       |john.smith@abc.com   |VALID       |
|1002       |mary.johnson@abc.com |VALID       |
|1003       |david.brown@abc.com  |VALID       |
|1004       |lisa.wilson@abc.com  |VALID       |
|1006       |robert.taylor@abc.com|VALID       |
|1007       |not_available        |INVALID     |
|1008       |peter.adams@abc.com  |VALID       |
+-----------+---------------------+------------+



In [67]:
print(" CUSTOMER NAME VALIDATION ")

clean_df.select(
    "customer_id",
    "customer_name"
).show(truncate=False)

 CUSTOMER NAME VALIDATION 
+-----------+-------------+
|customer_id|customer_name|
+-----------+-------------+
|1001       |John Smith   |
|1002       |MARY JOHNSON |
|1003       |David Brown  |
|1004       |Lisa Wilson  |
|1006       |Robert Taylor|
|1007       |Susan Lee    |
|1008       |Peter Adams  |
+-----------+-------------+



In [68]:
print("Original Records:", df.count())

print(
    "Original Unique Customer IDs:",
    df.select("customer_id").distinct().count()
)

print("Cleaned Records:", clean_df.count())

print(
    "Cleaned Unique Customer IDs:",
    clean_df.select("customer_id").distinct().count()
)

Original Records: 9
Original Unique Customer IDs: 8
Cleaned Records: 7
Cleaned Unique Customer IDs: 7
